## EDA: Consumos de Rega — `consumos_rega_89_22.csv`

Caminho do dataset:
`/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data/consumos_rega_89_22.csv`

Objetivo: analisar séries anuais por município (estatísticas, outliers, frequência anual e lacunas), localização e export de resumos — tudo dentro de `EDA`.


### 1) Setup de bibliotecas e configuração de paths


In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

DATA_FILE = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data/consumos_rega_89_22.csv")
assert DATA_FILE.exists(), f"CSV not found: {DATA_FILE}"

pd.options.display.float_format = "{:.3f}".format
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"Using file: {DATA_FILE}")


### 2) Leitura do CSV e preparação de colunas

#### 2.1 Explicação do código
- `ano` como data anual.
- Variáveis: `volume_rega_m3`, `volume_rega_hm3`.
- `site_id`/`site_label` por município e coordenadas.


In [ ]:
df = pd.read_csv(
    DATA_FILE,
    dtype={
        "municipio": "string",
        "volume_rega_m3": "float64",
        "volume_rega_hm3": "float64",
        "lat": "float32",
        "long": "float32",
    },
)

# Criar coluna date a partir de ano (1º de janeiro de cada ano)
df["date"] = pd.to_datetime(df["ano"].astype(int).astype(str) + "-01-01")

df = df.sort_values(["municipio", "date"]).reset_index(drop=True)

df["site_id"] = df["municipio"].str.replace(" ", "_", regex=False) + "_" + df["lat"].round(3).astype(str) + "_" + df["long"].round(3).astype(str)
df["site_label"] = df["municipio"].astype(str) + " | " + df["site_id"]

print(df.head()); df.info()


### 3) Cobertura temporal, frequência anual e lacunas por município

- Estatísticas dos volumes.
- Lacunas anuais por `site_id`.
- Duplicados por `ano`×`site_id`.


In [ ]:
print(df[["volume_rega_m3", "volume_rega_hm3"]].describe(percentiles=[0.01,0.05,0.95,0.99]))

dups = df[df.duplicated(["ano","site_id"], keep=False)]
print(f"Duplicate year entries (per site): {dups.shape[0]}")

def compute_missing_years(df_site: pd.DataFrame) -> tuple[int, list]:
    if df_site.empty:
        return 0, []
    years = pd.Index(df_site["date"].dt.year.unique())
    full = pd.Index(range(years.min(), years.max()+1))
    missing = full.difference(years)
    return int(missing.size), list(missing[:5].astype(int))

rows = []
for sid, d in df.groupby("site_id"):
    mcount, mfirst = compute_missing_years(d)
    rows.append({
        "site_id": sid,
        "site_label": d["site_label"].iloc[0],
        "freq_label": "annual",
        "missing_count": mcount,
        "missing_first_5": ", ".join(map(str, mfirst)),
    })
site_frequency_summary = pd.DataFrame(rows)
display(site_frequency_summary.head())


### 4) Outliers e gráficos

- Outliers (IQR + Z-score) para `volume_rega_hm3`.
- Gráfico agregado e dropdown por município.


In [ ]:
target_col = "volume_rega_hm3"

def detect_outliers_iqr(series: pd.Series, factor: float = 1.5) -> pd.Series:
    q1 = np.nanpercentile(series, 25); q3 = np.nanpercentile(series, 75)
    iqr = q3 - q1; lower = q1 - factor * iqr; upper = q3 + factor * iqr
    return (series < lower) | (series > upper)


def detect_outliers_zscore(series: pd.Series, threshold: float = 3.0) -> pd.Series:
    mu = np.nanmean(series); sigma = np.nanstd(series)
    if sigma == 0 or np.isnan(sigma):
        return pd.Series(False, index=series.index)
    z = (series - mu) / sigma; return z.abs() > threshold

flags = detect_outliers_iqr(df[target_col]) | detect_outliers_zscore(df[target_col])
df["is_outlier_rega"] = flags
print(f"{target_col} outliers: {flags.sum()}")

plt.figure(figsize=(14,5))
plt.scatter(df["date"], df[target_col], alpha=0.7, label="ALL SITES · volume_rega_hm3")
plt.scatter(df.loc[flags, "date"], df.loc[flags, target_col], s=20, color="black", label="outlier")
plt.legend(); plt.xlabel("date"); plt.ylabel("volume_rega_hm3 [hm3]")
plt.title("Consumos de rega (todas as observações)"); plt.tight_layout(); plt.show()

import ipywidgets as widgets
from IPython.display import display, clear_output
sites = sorted(df["site_id"].unique()); labels = df.groupby("site_id")["site_label"].first().to_dict()
opts = [(labels[s], s) for s in sites]; out = widgets.Output(); dd = widgets.Dropdown(options=opts, value=sites[0], description="município:")

def plot_site(sid: str):
    d = df[df["site_id"] == sid].sort_values("date"); lab = labels.get(sid, sid)
    plt.figure(figsize=(14,5))
    plt.plot(d["date"], d[target_col], marker="o", label=lab)
    if "is_outlier_rega" in d.columns:
        m = d["is_outlier_rega"]; plt.scatter(d.loc[m, "date"], d.loc[m, target_col], s=20, color="black", label="outlier")
    plt.legend(); plt.xlabel("date"); plt.ylabel("volume_rega_hm3 [hm3]"); plt.title(f"Consumos de rega ({lab})")
    plt.tight_layout(); plt.show()

def on_change(c):
    if c["name"] == "value":
        with out:
            clear_output(wait=True); plot_site(c["new"]) 

dd.observe(on_change, names="value"); display(dd)
with out: plot_site(dd.value)
display(out)


### 5) Resumo por município e export

- Junta frequência e lacunas por `site_id`.
- Export para `EDA /scripts/consumos_rega/resources/`.


In [ ]:
location_summary = (
    df.groupby(["site_id", "site_label", "lat", "long"]).agg(
        n_obs=("ano", "count"),
        first_year=("ano", "min"),
        last_year=("ano", "max"),
        mean_hm3=("volume_rega_hm3", "mean"),
        p95_hm3=("volume_rega_hm3", lambda s: np.nanpercentile(s, 95)),
    ).reset_index()
)

location_summary = location_summary.merge(site_frequency_summary, on=["site_id", "site_label"], how="left")

display(location_summary.head())

from pathlib import Path
out_dir = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /scripts/consumos_rega/resources")
out_dir.mkdir(parents=True, exist_ok=True)

general_stats = df[["volume_rega_hm3"]].describe().T
general_stats.loc["meta_unique_points", "count"] = location_summary.shape[0]

location_summary.to_csv(out_dir / "consumos_rega_location_summary.csv", index=False)
site_frequency_summary.to_csv(out_dir / "consumos_rega_frequency_gaps_by_site.csv", index=False)
general_stats.to_csv(out_dir / "consumos_rega_summary_stats.csv")

print(f"Saved summaries in: {out_dir}")
